# Home Affairs: Temporary Visa Holders in the Workforce

Estimates the stock of temporary visa holders in the Australian workforce, by visa class.
Combines Home Affairs quarterly visa stock data with ABS ACTEID 2021 (Census-linked)
employment rates by visa class. Also plots NOM by visa pathway from ABS 3407.0.

**Sources**
- Home Affairs *Temporary visa holders in Australia* (data.gov.au, BP0019, quarterly)
- ABS *Temporary visa holders in Australia, 2021* (ACTEID, employment rate by class)
- ABS *Overseas Migration, 2024-25* (3407.0, Table 4.1)

## Setup

In [1]:
# Standard library
import json
import urllib.request
from pathlib import Path

# Third party
import pandas as pd

# Local
import mgplot as mg

In [2]:
# Pandas display
pd.options.display.max_rows = 999999

# Chart output
CHART_DIR = './CHARTS/Home Affairs - Temporary Visa Holders/'
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()
SHOW = False
SOURCE_HA = 'Source: Home Affairs (data.gov.au); ABS ACTEID 2021'
SOURCE_ABS = 'Source: ABS Overseas Migration 3407.0, Table 4.1'

# Cache for downloaded source files
CACHE_DIR = Path('./CACHE/HomeAffairs')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Data sources
PACKAGE_SHOW_URL = (
    'https://data.gov.au/data/api/3/action/package_show'
    '?id=temporary-entrants-visa-holders'
)
ABS_MIGRATION_URL = (
    'https://www.abs.gov.au/statistics/people/population/overseas-migration/'
    '2024-25/34070DO004_202425.xlsx'
)
ABS_MIGRATION_FILE = '34070DO004_202425.xlsx'

In [3]:
# ACTEID Census 2021 employment rates for visa-holders aged 15+
# Applied to current Home Affairs visa-stock counts as a proxy for working share.
EMPLOYMENT_RATE = {
    'Bridging':                                0.674,  # Other-temp rate
    'Crew and Transit':                        0.0,    # No work rights
    'Other Temporary':                         0.674,
    'Special Category':                        0.679,  # NZ SCV (444)
    'Student':                                 0.636,
    'Temporary Graduate':                      0.674,
    'Temporary Protection':                    0.674,
    'Temporary Resident (Other Employment)':   0.843,  # Temp-skilled rate
    'Temporary Resident (Skilled Employment)': 0.843,
    'Visitor':                                 0.0,    # No work rights
    'Working Holiday Maker':                   0.857,
}

# Plot order and friendly labels for the working-stock chart
PLOT_ORDER = [
    'Special Category',
    'Student',
    'Working Holiday Maker',
    'Temporary Resident (Skilled Employment)',
    'Bridging',
    'Temporary Graduate',
    'Temporary Resident (Other Employment)',
    'Other Temporary',
    'Temporary Protection',
]

LABEL = {
    'Special Category':                        'NZ SCV (444)',
    'Student':                                 'Student (500)',
    'Working Holiday Maker':                   'Working Holiday (417/462)',
    'Temporary Resident (Skilled Employment)': 'Skilled (482/SID/186)',
    'Bridging':                                'Bridging',
    'Temporary Graduate':                      'Graduate (485)',
    'Temporary Resident (Other Employment)':   'Other Employment',
    'Other Temporary':                         'Other Temporary',
    'Temporary Protection':                    'Temporary Protection',
}

## Helper functions

In [4]:
def _download(url: str, filename: str) -> Path:
    """Fetch url to CACHE_DIR/filename (cached on subsequent runs)."""
    cache_path = CACHE_DIR / filename
    if cache_path.exists():
        print(f'Using cached: {filename}')
        return cache_path
    print(f'Downloading: {filename}')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:  # noqa: S310
        cache_path.write_bytes(r.read())
    return cache_path


def fetch_current_resource_url() -> str:
    """Query data.gov.au CKAN API for the current BP0019 XLSX URL."""
    req = urllib.request.Request(
        PACKAGE_SHOW_URL,
        headers={'Accept': 'application/json', 'User-Agent': 'Mozilla/5.0'},
    )
    with urllib.request.urlopen(req, timeout=30) as r:  # noqa: S310
        payload = json.load(r)
    for res in payload['result']['resources']:
        if (res.get('format', '').upper() == 'XLSX'
                and 'bp0019' in res.get('url', '').lower()):
            return res['url']
    raise RuntimeError('No BP0019 XLSX resource found in dataset')

In [5]:
def fetch_visa_stock() -> pd.DataFrame:
    """Download Home Affairs temp visa XLSX, return quarterly visa stock by class.

    Returns: DataFrame with quarterly PeriodIndex rows and visa-category columns.
    """
    url = fetch_current_resource_url()
    filename = url.rsplit('/', 1)[-1]
    cache_path = _download(url, filename)

    raw = pd.read_excel(cache_path, sheet_name='Visa Holders', header=None)

    # Row 9 has date headers; col 0 has 'Visa Category'; data rows are 10..N
    dates = pd.to_datetime(raw.iloc[9, 1:].tolist())
    categories = raw.iloc[10:, 0].astype(str).tolist()
    values = raw.iloc[10:, 1:].apply(pd.to_numeric, errors='coerce')

    df = pd.DataFrame(values.values, index=categories, columns=dates)
    df.index.name = 'Visa Category'
    df = df.drop(index='Grand Total', errors='ignore').T
    df.index = df.index.to_period('Q')
    return df

In [6]:
def _parse_migration_block(block: pd.DataFrame, years: list[str]) -> pd.DataFrame:
    """Extract an ABS Migration arrivals/departures block, keyed by most-detailed label."""
    labels = block.iloc[:, 2].where(block.iloc[:, 2].notna(), block.iloc[:, 1])
    keep = labels.notna() & ~labels.astype(str).str.startswith('Total')
    df = pd.DataFrame(
        block.iloc[:, 3:].apply(pd.to_numeric, errors='coerce').values,
        index=labels.values,
        columns=years,
    )
    return df.loc[keep.values].dropna(how='all')


def fetch_nom_by_visa() -> pd.DataFrame:
    """ABS 3407.0 Table 4.1: NOM by visa pathway, financial years 2004-05 onwards."""
    cache_path = _download(ABS_MIGRATION_URL, ABS_MIGRATION_FILE)
    raw = pd.read_excel(cache_path, sheet_name='Table 4.1', header=None)

    years = (
        raw.iloc[15, 3:].astype(str)
        .str.replace(r'\(.*\)', '', regex=True)
        .str.strip().tolist()
    )
    arrivals = _parse_migration_block(raw.iloc[16:33, :], years)
    departures = _parse_migration_block(raw.iloc[34:51, :], years)
    common = arrivals.index.intersection(departures.index)
    nom = arrivals.loc[common] - departures.loc[common]

    # Consolidate to plotting pathways
    out = pd.DataFrame(index=nom.columns)
    out['Permanent: Skilled'] = nom.loc['Skilled (permanent)']
    out['Permanent: Family/Hum/Other'] = nom.loc[
        ['Family', 'Special eligibility & humanitarian', 'Other (permanent)']
    ].sum()
    out['Temporary: Student'] = nom.loc['Student']
    out['Temporary: Skilled'] = nom.loc['Skilled (temporary)']
    out['Temporary: Working holiday'] = nom.loc['Working holiday']
    out['Temporary: Visitors/Other'] = nom.loc[['Visitors', 'Other (temporary)']].sum()
    out['NZ citizens (444)'] = nom.loc['New Zealand citizens (subclass 444)(h)']
    return out


def compute_workforce(visa_stock: pd.DataFrame) -> pd.DataFrame:
    """Apply visa-class employment rates to the visa stock."""
    working = visa_stock.copy()
    for col in working.columns:
        working[col] = working[col] * EMPLOYMENT_RATE.get(col, 0.0)
    return working

## Plotting functions

In [7]:
STACK_COLORS = [
    '#1f77b4',  # NZ
    '#ff7f0e',  # Student
    '#2ca02c',  # WHM
    '#d62728',  # Skilled
    '#9467bd',  # Bridging
    '#8c564b',  # Graduate
    '#e377c2',  # Other Employment
    '#7f7f7f',  # Other Temp
    '#bcbd22',  # Protection
]

NOM_COLORS = [
    '#1a6fbf',  # Perm skilled
    '#a6cee3',  # Perm family/hum
    '#ff7f0e',  # Student
    '#d62728',  # Temp skilled
    '#2ca02c',  # WHM
    '#7f7f7f',  # Visitors/Other temp
    '#9467bd',  # NZ
]


def plot_working_stock_by_class(working: pd.DataFrame) -> None:
    plot_df = (working[PLOT_ORDER].rename(columns=LABEL) / 1_000)
    mg.bar_plot_finalise(
        plot_df,
        stacked=True, annotate=False, width=0.9, color=STACK_COLORS,
        title='Temporary visa holders in the Australian workforce, by visa class',
        ylabel="Number ('000s)",
        rfooter=SOURCE_HA,
        lfooter='Australia. Visa stock x ABS ACTEID 2021 employment rate by class.',
        legend={'loc': 'upper left', 'fontsize': 'x-small', 'ncol': 2},
        show=SHOW,
    )


def plot_working_stock_total(working: pd.DataFrame) -> None:
    total = working.sum(axis=1) / 1_000
    total.name = 'Total'
    mg.line_plot_finalise(
        total,
        width=2.5, color=['navy'], annotate=True, rounding=0,
        title='Estimated temporary visa holders in the Australian workforce: total',
        ylabel="Number ('000s)",
        rfooter=SOURCE_HA,
        lfooter='Australia. Visa stock x ACTEID 2021 employment rate by class.',
        legend=False, show=SHOW,
    )


def plot_nom_composition(nom: pd.DataFrame) -> None:
    plot_df = nom / 1_000
    plot_df.index.name = 'Year'
    mg.bar_plot_finalise(
        plot_df,
        stacked=True, annotate=False, width=0.9, label_rotation=45, color=NOM_COLORS,
        title='Australia: Net overseas migration by visa pathway',
        ylabel="Persons ('000s)",
        rfooter=SOURCE_ABS,
        lfooter='Australia. Financial years. 12/16-month rule basis. NOM = arrivals - departures.',
        legend={'loc': 'upper left', 'fontsize': 'x-small', 'ncol': 2},
        y0=True, show=SHOW,
    )

## Fetch data

In [8]:
visa_stock = fetch_visa_stock()
working = compute_workforce(visa_stock)
nom = fetch_nom_by_visa()

Downloading: bp0019l-number-of-temporary-visa-holders-in-australia-at-2026-04-30-v100.xlsx


Using cached: 34070DO004_202425.xlsx


## Headline numbers

In [9]:
latest = visa_stock.index[-1]
print(f'Visa stock at {latest}:')
print(visa_stock.iloc[-1].sort_values(ascending=False).map(lambda x: f'{x:>12,.0f}'))
print()
print(f'Estimated working stock at {latest}:')
print(working.iloc[-1].sort_values(ascending=False).map(lambda x: f'{x:>12,.0f}'))
print()
print(f'Total visa stock:    {visa_stock.iloc[-1].sum():>14,.0f}')
print(f'Total working stock: {working.iloc[-1].sum():>14,.0f}')

Visa stock at 2026Q2:
Visa Category
Special Category                                735,295
Student                                         609,984
Bridging                                        419,682
Visitor                                         318,350
Temporary Graduate                              266,921
Temporary Resident (Skilled Employment)         259,665
Working Holiday Maker                           236,457
Temporary Resident (Other Employment)            62,422
Crew and Transit                                 12,708
Other Temporary                                   5,598
Temporary Protection                              1,737
Name: 2026Q2, dtype: str

Estimated working stock at 2026Q2:
Visa Category
Special Category                                499,265
Student                                         387,950
Bridging                                        282,866
Temporary Resident (Skilled Employment)         218,898
Working Holiday Maker                           2

## Charts

In [10]:
plot_working_stock_by_class(working)
plot_working_stock_total(working)
plot_nom_composition(nom)

## Watermark

In [11]:
%load_ext watermark
%watermark -u -t -d -v -iv -m

Last updated: 2026-06-24 11:03:19

Python implementation: CPython
Python version       : 3.14.2
IPython version      : 9.14.1

Compiler    : Clang 21.1.4 
OS          : Darwin
Release     : 25.5.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

json   : 2.0.9
mgplot : 0.2.28
pandas : 3.0.3
pathlib: 1.0.1

